[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saghiles/semantic-id-tutorial/blob/main/session1_part2_tiger_vs_sasrec.ipynb)

# Hands-on 1 — Part 2: TIGER (Semantic IDs) vs. SASRec

## Goal

In Part 1 we trained an RQ-KMeans tokenizer on item text embeddings and built
the **Semantic IDs (SIDs)** used by TIGER. Training the full TIGER model
(tokenizer + encoder-decoder) end to end, together with SASRec, takes longer
than we have in the room, so both models were **trained ahead of time** with
the companion script `train_and_save_models.py` (also provided, in case you
want to retrain or tune them yourself afterwards).

In this notebook we:

1. **Load** the pretrained SASRec and TIGER checkpoints, no training here.
2. **Load** the Beauty interactions and **rebuild the same train/val/test
   split** used at training time (same seed, so the split is identical).
3. **Evaluate** both loaded models with the same next-item protocol
   (Recall@10, NDCG@10, MRR) to compare an ID-based recommender (SASRec)
   with a generative, SID-based one (TIGER).
4. **Compare** model size and number of parameters.
5. **Play with TIGER**: inspect SIDs, feed a history taken from the test set
   to the trained model, look at the SIDs it generates, and map them back to
   items.
6. **Take-home exercise**: use `train_and_save_models.py` to tune TIGER's
   hyperparameters and try to match the original paper's numbers.

| | SASRec (baseline) | TIGER (generative retrieval) |
|---|---|---|
| Item representation | one learned embedding **per item** (atomic ID) | a short tuple of discrete codes, the **Semantic ID (SID)** |
| Prediction | score every item in the catalog | **generate** the SID of the next item token by token |
| Item content used? | no | yes, through text embeddings quantized into SIDs (Part 1) |

## Installation

In [ ]:
# --- Colab setup (skipped automatically when running locally) ---
import os, sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/saghiles/semantic-id-tutorial.git"
    REPO_DIR = "/content/semantic-id-tutorial"
    # the item features / saved models are stored with Git LFS
    !command -v git-lfs >/dev/null || apt-get install -y -qq git-lfs
    !git lfs install
    if not os.path.exists(REPO_DIR):     # re-running this cell is safe
        # clone the pointer files first, then fetch the real LFS files so any
        # LFS download error is shown instead of silently aborting the checkout
        !GIT_LFS_SKIP_SMUDGE=1 git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git lfs pull
    !pip install -q cornac==3.0.0
    !ls -lh item_features

## 1. Imports

In [ ]:
import random

import torch

from cornac.eval_methods import NextItemEvaluation
from cornac.datasets import amazon_review
from cornac.experiment import Experiment
from cornac.metrics import MRR, NDCG, Recall
from cornac.models import SASRec, TIGER

SAVE_DIR = "saved_models"
SPLIT_SEED = 123  # must match the seed used by train_and_save_models.py


## 2. Load the pretrained models

`trainable=False` marks the models as fixed: nothing here retrains the
RQ-KMeans tokenizer or the TIGER seq2seq model — we're reusing what
`train_and_save_models.py` already produced.


In [ ]:
sasrec = SASRec.load(f"{SAVE_DIR}/SASRec", trainable=False)
tiger = TIGER.load(f"{SAVE_DIR}/TIGER", trainable=False)

print(f"Loaded SASRec ({sasrec.num_items} items)")
print(f"Loaded TIGER ({tiger.num_items} items, {len(tiger.sid_table)} SIDs)")


## 3. Load the Beauty interactions and item text

Same raw data as Part 1 and as `train_and_save_models.py`. We only need the
*interactions* to rebuild the evaluation split; `texts`/`id2text` are kept
for display purposes in the TIGER playground later on.


In [ ]:
data = amazon_review.load_feedback(category="beauty")           # (user, item, rating, timestamp)
texts, item_ids = amazon_review.load_text(category="beauty")    # item text, aligned with item_ids
id2text = dict(zip(item_ids, texts))


## 4. Rebuild the same train/val/test split

`leave_last_out` is deterministic given the same data and seed (it holds out
the last interaction per user chronologically), so reusing `SPLIT_SEED`
reproduces the exact split the models were trained and validated on, no
need to re-encode item text or save the split itself.


In [ ]:
next_item_eval = NextItemEvaluation.leave_last_out(
    data=data,
    exclude_unknowns=True,
    verbose=True,
    seed=SPLIT_SEED,  # same seed -> identical train/val/test split
)


## 5. Evaluate both loaded models

Same metrics as the original training run: Recall@10, NDCG@10, MRR.


In [ ]:
metrics = [Recall(k=10), NDCG(k=10), MRR()]

Experiment(
    eval_method=next_item_eval,
    models=[sasrec, tiger],
    metrics=metrics,
    show_validation=False,
).run()


## 6. Model size and number of parameters

SASRec needs one embedding row per item, so its size grows linearly with the
catalog. TIGER only needs embeddings for the (small) SID vocabulary.


In [ ]:
def report_model_size(name, torch_model):
    """Print number of parameters and memory footprint of a torch module."""
    n_params = sum(p.numel() for p in torch_model.parameters())
    size_bytes = sum(p.numel() * p.element_size() for p in torch_model.parameters())
    print(f"{name} model size: {size_bytes / 1024**2:.2f} MB")
    print(f"{name} number of parameters: {n_params:,}")


report_model_size("SASRec", sasrec.model)
report_model_size("TIGER", tiger.model)


## 7. Play with TIGER

Now let's open the black box:

1. **Inspect the SIDs**: each item is a tuple `(c1, c2, c3, dedup)`, three
   RQ-KMeans codes plus an extra level that disambiguates items sharing the
   same codes.
2. **Generate from a 2-item history** taken from the test set and compare
   with the ground-truth next item.
3. **Generate from a single item**, then map the generated SID back to an
   item.

Generation uses *constrained beam search*: at each level only tokens that
lead to a valid SID prefix are allowed, so every generated SID corresponds
to a real item.


### 7.1 Helpers for the playground

In [ ]:
train_set = next_item_eval.train_set
test_set = next_item_eval.test_set
idx2raw = {idx: raw for raw, idx in train_set.iid_map.items()}  # internal item index -> raw item id
N_KNOWN_ITEMS = train_set.num_items                              # items that have a SID


def item_sid(idx):
    """SID of an item (internal index) as a tuple of ints."""
    return tuple(int(c) for c in tiger.sid_table[idx])


def item_text(idx, max_chars=90):
    """Short text description of an item (internal index)."""
    return id2text.get(idx2raw[idx], "<no text>")[:max_chars].replace("\n", " ")


def show_item(idx, prefix=""):
    print(f"{prefix}item {idx:>6} | SID {item_sid(idx)} | {item_text(idx)}")


@torch.no_grad()
def generate_sids(history, n_beams=10):
    """Run TIGER's constrained beam search on a history of item indices.

    Returns a list of (sid_tuple, log_prob, item_idx) sorted by decreasing log-prob.
    Mirrors what TIGER.score() does internally, but exposes the generated SIDs.
    """
    history = list(history)[-tiger.max_len:]
    history = [tiger.pad_idx] * (tiger.max_len - len(history)) + history  # left-pad
    tiger._ensure_device(torch)
    enc_tokens = torch.tensor(
        tiger.enc_token_table[history].reshape(1, -1), dtype=torch.long, device=tiger.device_
    )
    enc_mask = (enc_tokens != 0).float()  # 0 = padding token
    tiger.model.eval()
    beams, logps = tiger.model.generate_beam(enc_tokens, enc_mask, n_beams, tiger.prefix_children)
    return [(sid, float(lp), tiger.sid_to_item[sid]) for sid, lp in zip(beams, logps)]


def shared_prefix_len(a, b):
    """Number of leading SID levels shared by two SIDs."""
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


### 7.2 Pick a test session and look at its SIDs

In [ ]:
random.seed(0)
candidates = []
for [sid], [mapped_ids], [session_items] in test_set.si_iter(batch_size=1, shuffle=False):
    # need >= 3 items: 2 for the input history + 1 ground-truth next item, all with a SID
    if len(session_items) >= 3 and all(i < N_KNOWN_ITEMS for i in session_items[:3]):
        candidates.append(list(session_items))
session = random.choice(candidates)

print("Number of items with a SID:", N_KNOWN_ITEMS)
print("Vocabulary size per SID level:", tiger.level_sizes)


### 7.3 Investigation A: two-item history → generate the next SID

In [ ]:
history, ground_truth = session[:2], session[2]

print("=== Input history (2 items) ===")
for idx in history:
    show_item(idx, prefix="  ")
print("\n=== Ground-truth next item ===")
show_item(ground_truth, prefix="  ")

print("\n=== TIGER top-5 generated SIDs -> items ===")
for rank, (sid, logp, idx) in enumerate(generate_sids(history, n_beams=10)[:5], start=1):
    hit = "  <-- ground truth" if idx == ground_truth else ""
    print(f"#{rank} logp={logp:6.2f} | SID {sid} | {item_text(idx, 70)}{hit}")


### 7.4 Investigation B: single-item history → generate one SID, then look the item up

In [ ]:
seed_item = history[-1]  # re-use the last item of the previous history, or change it!
print("=== Input item ===")
show_item(seed_item, prefix="  ")

(top_sid, top_logp, top_item), *_ = generate_sids([seed_item], n_beams=10)
print(f"\nTop generated SID: {top_sid} (logp={top_logp:.2f})")
print("SID -> item lookup via tiger.sid_to_item:")
show_item(tiger.sid_to_item[top_sid], prefix="  ")


**Things to try**
* Change the seed item or the session and see how the generated SIDs change.
* Increase `n_beams` and check whether the ground-truth item shows up further down the list.
* Compare the text of the input item with the generated items: how many SID levels are shared?
* Find an item whose SID has `dedup > 0` (a collision) and look at its "twin" item.


## 8. Take-home exercise

Using `train_and_save_models.py`, play with TIGER's hyperparameters (depth,
number of heads, learning rate, batch size, tokenizer, ...) and try to match
the performance reported in the original paper on the Beauty dataset.
**Target: Recall@10 ~ 0.065.**
